# 05 — L'expérience oracle

Le notebook 02 a posé le wrapper $\psi$/$\bar A$/$\pi$, l'orbite de jauge qui le rend
identifiable à une transformation près, et le budget qui compare les bras à paramètres
égaux. Le notebook 03 a câblé la loss et défini $e_H$, le VPT et la porte de contrôle
positif. Le notebook 04, en cours d'écriture, pose le diagnostic sur lequel ce chapitre-ci
ferme l'histoire : sept racines ont mesuré, sur `mlp` et sur le minimal causal transformer,
la même chose — en base identité, la classe structurée coûte plus qu'elle ne rapporte.
`PROTOCOL.md` §26.0 a ensuite montré, en forme close, que le prior spectral est exact
**dans une base**, et que cette base n'est pas l'identité : la pénalité pourrait venir de la
découverte **jointe** de la base et de l'opérateur, pas de la classe structurée elle-même.

Ce chapitre exécute l'expérience qui teste ce diagnostic : donner la base au lieu de la
faire découvrir par la descente de gradient, et regarder si la pénalité disparaît.

## Plan

1. Le problème — pourquoi cette expérience a été refusée une fois, puis autorisée.
2. Geler une base est une décision, pas un calcul.
3. Le contrat, écrit avant les nombres.
4. La preuve vivante — la base rediagonalise le générateur vrai, le mutant la casse, les
   quinze verdicts recalculés.
5. Le résultat.
6. La figure.
7. Ce qui a été prouvé, la limite, et le pont.

**Sources.** `PROTOCOL.md` §26.0, §27.9, §28 (Amendment 18) en entier ;
`src/sld/experiments/configs/basis_root.yaml` ; `tests/test_basis_root.py` ;
`src/sld/viz/figH_basis_oracle.py` ; les quinze `gate.json` de
`results_basis/basis_root/`.


In [ ]:
from pathlib import Path
import hashlib

import numpy as np
import yaml
from IPython.display import Image, display

from sld.systems.coupled_oscillators import CoupledOscillators
from sld.systems.representability import representability_distance
from sld.viz.figH_basis_oracle import make_figH
from sld.viz.ingest import load_gate

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


## 1. Le problème

`PROTOCOL.md` §26.0 mesure, en forme close, qu'à `c = 0.10` — la configuration où
$\delta(c)$ vaut zéro, « le prior est exact » — le générateur vrai $A$ est malgré tout à
distance de Frobenius **1.0654** de la classe bloc-diagonale rotation-scaling **dans la
base identité**, soit 36 % de $\|A\|_F = 2.9614$. Dans la meilleure base — celle de sa
propre forme de Jordan réelle — cette distance tombe à **1.97e-15**. Le prior est exact,
mais dans une base qui n'est pas l'identité.

§26.0 en tire une hypothèse qu'elle refuse expressément de tester : *the penalty is neither
`psi`/`pi` alone nor the transition alone, but their COUPLING*. Elle s'arrête là parce que
le plafond de `PROTOCOL.md` §25.5 — deux racines, pas une de plus, sans justification
écrite — était déjà atteint. `PROTOCOL.md` §28.0 lève ce plafond, une fois, et écrit
l'argument avant la première cellule : la question posée par ce chapitre n'est pas une
question déjà répondue reposée avec un nouveau bouton — ce que le plafond existe pour
interdire — c'est la question que §26.0 a explicitement déclaré ne pas tester, pour une
raison de gouvernance et non scientifique. C'est la R4 en action : l'amendement qui
autorise l'exécution se committe avant elle, avec l'argument qu'un lecteur pourrait rejeter
écrit dans le même paragraphe.


## 2. Geler une base est une décision, pas un calcul

La base candidate est simple à énoncer : $T$ tel que $T A T^{-1}$ soit bloc-diagonal
rotation-scaling, c'est-à-dire la base de la forme de Jordan réelle de $A$. La calculer
semble immédiate — `numpy.linalg.eig(A)` donne les vecteurs propres. Mais `eig` ne garantit
ni leur **échelle** ni leur **ordre**, et rien ne garantit la stabilité de ces deux choix
entre deux implémentations LAPACK : recalculer $T$ sur une autre machine donnerait une
matrice différente réalisant la même base, tout en semblant déterministe.

La base n'est donc pas *calculée* à l'exécution : elle est **choisie une fois, gelée en
littéral, et empreinte**. `src/sld/experiments/configs/basis_root.yaml` porte $T$ comme un
tableau de seize flottants, avec son `sha256` inscrit en commentaire à côté ; ses propriétés
ne sont pas prises au mot — elles sont recalculées dans `tests/test_basis_root.py` à chaque
exécution de la suite plutôt que d'être seulement citées.


In [ ]:
cfg = yaml.safe_load((REPO / "src/sld/experiments/configs/basis_root.yaml").read_text())
oracle_arm = [a for a in cfg["arms"] if a["overrides"].get("lift") == "frozen"]
assert len(oracle_arm) == 1, "un seul bras oracle attendu"
T = np.array(oracle_arm[0]["overrides"]["psi_matrix"], dtype=float)

declared_sha256 = "e026f402194929b09fdb6c7c88fe801e5d945131daa537947cd13ac7b8736be5"
measured_sha256 = hashlib.sha256(T.tobytes()).hexdigest()
assert measured_sha256 == declared_sha256

print(f"T : forme {T.shape}")
print(f"sha256 mesure   : {measured_sha256}")
print(f"sha256 declare  : {declared_sha256}")


Le littéral chargé retrouve exactement le `sha256` inscrit dans `basis_root.yaml` — la
matrice utilisée par les quinze cellules de la racine est celle décrite en §28.2, pas une
approximation recalculée à la volée.

**Le fil de détente, retiré au bon moment.** `tests/test_basis_root.py` portait, jusqu'au
commit `ac80466` (« 28 : la racine demarre -- cout mesure sur une cellule on-oracle seule
(2186 s, 0.273 s/update), le fil de detente retire, results_basis/ ignore »), un test
`test_no_cell_has_run_yet` : une tripwire qui échouait si `results_basis/` contenait déjà un
`gate.json` avant que le coût d'une cellule `on-oracle` isolée n'ait été mesuré, comme
§28.2 l'exige. Le même commit qui enregistre ce coût (2186 s, 0.273 s/mise à jour) retire le
test — pas un oubli de nettoyage, mais la R4 vivante : un verrou qui se retire quand, et
seulement quand, la chose qu'il gardait devient vraie.

**Rappel d'une phrase.** Le théorème de jauge du notebook 02 (§3.2) montre que
$T \bar A T^{-1} = \bar A$ : la transition est un point fixe de l'orbite qui laisse la
sortie inchangée — c'est cette même invariance qui garantit que geler $\psi = T$ ne change
rien d'autre à la dynamique que l'accès à sa base.


## 3. Le contrat, écrit avant les nombres

`results_basis/`, **15 cellules, trois bras × cinq graines** (`PROTOCOL.md` §28.2). Le
stratum est celui de 27.1, repris verbatim pour que les contrastes restent commensurables
avec §27.8.3 : `CoupledOscillators`, `OURegime(theta_ou=1.0)`, `c=0.10`, `n=32`, `h_fr=25`,
`H=25`, budget de 8000 mises à jour, représentation identité partagée, taux `1e-3`.

**Un seul backbone : `mlp`.** §27.9 laisse `mlp` comme la colonne où le négatif est porté
par 5 seeds sur 5 avec l'écart-type le plus serré (0.0351) et la provenance de taux la plus
solide — le minimal causal transformer, l'autre backbone où ce négatif tient, n'est pas
repris ici. Un mécanisme se teste là où l'effet à expliquer est le mieux mesuré, pas là où
la mesure est la plus faible.

**Les trois bras :** `off` (non structuré) ; `on-learned` (structuré, $\psi$/$\pi$ appris
conjointement avec l'opérateur, exactement le `mlp_k` de §27) ; `on-oracle` (structuré,
$\psi = T$, $\pi = T^{-1}$, **gelés, sans gradient**).

**Le budget est suspendu par nom, pas appliqué** — rappel d'une phrase du notebook 02
(§4.2) : `on-oracle` hérite la largeur et `psi_hidden` de `on-learned` à l'identique,
plutôt que de laisser la recherche lui attribuer un backbone plus large pour compenser les
paramètres entraînables perdus.

**Les trois prédictions**, signe prédit avant tout chiffre : P1 `on-learned > off` (la
structure coûte quand la base doit être apprise — un contrôle, la réplique de §27.8.3) ;
P2 `on-oracle < on-learned` (donner la base aide le bras structuré — le candidat de
couplage de §26.0, énoncé comme un signe mesurable) ; P3 `on-oracle < off` (avec la base
donnée, la classe structurée bat enfin la classe non structurée — la prétention forte).


### Ce que chaque issue est autorisée à dire, fixé avant de connaître laquelle se réalise

`PROTOCOL.md` §28.4 écrit la phrase de chaque issue possible avant le premier run — la
conclusion n'est donc pas rédigée après coup, elle est choisie dans une liste écrite
d'avance :

| issue | ce qui est publié |
|---|---|
| P2 `contradicted` ou `indeterminate` | donner la base n'aide pas ; le candidat de couplage de §26.0 est réfuté comme explication de la pénalité |
| P2 `supported`, P3 `contradicted` ou `indeterminate` | la base est une part réelle de la pénalité, pas toute la pénalité — jamais « la structure marche quand on fixe la base » |
| P2 et P3 `supported` | le diagnostic tient de bout en bout, borné par le mot **oracle** : la classe est atteignable quand la base est donnée, sur un système, un backbone |
| P1 non répliqué | la racine est **ininterprétable**, P2 et P3 ne se lisent pas du tout |


## 4. La preuve vivante

Le générateur vrai $A$ et sa distance à la classe bloc-diagonale rotation-scaling ont été
prouvés et paritisés au notebook 00 (§3.3-§3.4) : `representability_distance` y est dérivée,
implémentée à la main, puis confrontée à `sld`. Ici la question n'est plus de dériver cette
distance mais de vérifier ce que le littéral $T$ de `basis_root.yaml` en fait réellement,
sur l'objet que la campagne a effectivement tourné.


In [ ]:
A = CoupledOscillators(c=0.10).A_mat
M = T @ A @ np.linalg.inv(T)
distance, pairing = representability_distance(M)

off_block = max(np.abs(M[:2, 2:]).max(), np.abs(M[2:, :2]).max())
print(f"distance a la classe        : {distance:.3e}  (pairing {pairing})")
print(f"plus grande entree hors-bloc: {off_block:.3e}")
print(f"bloc 1 : alpha={M[0, 0]:+.4f}, omega={M[0, 1]:+.4f}")
print(f"bloc 2 : alpha={M[2, 2]:+.4f}, omega={M[2, 3]:+.4f}")

assert distance < 1e-12
assert off_block < 1e-12


Le générateur vrai, reconjugué par le littéral de `basis_root.yaml`, retombe bloc-diagonal
rotation-scaling à **1.3e-15** — sous la tolérance de $10^{-12}$ que `tests/test_basis_root.py`
exige. Les deux blocs retrouvés, `(-0.05, ±1.5187)` et `(-0.05, ±1.0902)`, sont exactement
les deux paires que §26.0 a nommées. `T` n'est donc pas une base plausible : c'est la base
qui diagonalise ce générateur précis, au flottant près.


**Le mutant.** `REGLES.md` §7.1 exige que tout objet mathématique porte un test positif
*et* un test négatif — un check qui ne s'allume jamais sur une base cassée ne prouve rien
sur celles qui ne le sont pas. La cellule suivante casse $T$ en couplant les deux
sous-espaces 2×2 que la vraie base garde disjoints : `T[0, 2] += 0.1` mélange le bloc
`{0, 1}` dans le bloc `{2, 3}`.


In [ ]:
T_mutant = T.copy()
T_mutant[0, 2] += 0.1
M_mutant = T_mutant @ A @ np.linalg.inv(T_mutant)
distance_mutant, _ = representability_distance(M_mutant)

print(f"distance de la base cassee : {distance_mutant:.5f}")
assert distance_mutant > 1e-6


La distance saute de **1.3e-15** à **≈0.1546** — cinq ordres de grandeur, pour une
perturbation d'une seule entrée sur seize. `tests/test_basis_root.py` republie cette même
valeur (0.15462926506...) et l'apparie à un second test qui montre le sens inverse : une
remise à l'échelle bénigne par bloc (`diag(3, 3, 0.2, 0.2) @ T`) laisse la distance sous
1e-12 alors qu'elle change l'échelle du latent d'un facteur 2.1. Le check ne s'allume ni
trop tôt (la remise à l'échelle bénigne) ni trop tard (ce mutant) — c'est le critère exact
de §7.1, « une fausse implémentation échouerait-elle à ce test ».


### Les quinze verdicts, recalculés

`results_basis/basis_root/` porte quinze `gate.json`, un par cellule. La racine est
gitignorée — absente d'un clone frais — donc la cellule suivante porte sa propre garde :
si elle manque, elle s'arrête proprement et les chiffres publiés en §5 restent cités contre
`PROTOCOL.md` §28.7, jamais recalculés dans le vide.


In [ ]:
results_root = REPO / "results_basis"

if not results_root.exists():
    print(
        f"[garde] {results_root} absente -- racine gitignoree, absente d'un clone frais. "
        "Les valeurs de la §5 restent citees en texte, contre PROTOCOL.md §28.7."
    )
    contrasts = {}
else:
    scalars, _ = load_gate(results_root)
    assert len(scalars) == 15

    def arm_of(structure: str, run_dir: str) -> str:
        if structure == "off":
            return "off"
        lift = yaml.safe_load((results_root / run_dir / "config.yaml").read_text()).get("lift")
        return "on-oracle" if lift == "frozen" else "on-learned"

    scalars = scalars.copy()
    scalars["arm"] = [arm_of(s, r) for s, r in zip(scalars["structure"], scalars["run_dir"])]

    piv = scalars.pivot(index="seed", columns="arm", values="e_h_gate").sort_index()
    means = piv.mean()
    control = scalars.groupby("arm")["status"].apply(lambda s: int((s == "learns").sum()))

    def contrast(a, b, sign):
        diffs = (piv[a] - piv[b]).to_numpy()
        mean, se = float(diffs.mean()), float(diffs.std(ddof=1) / np.sqrt(len(diffs)))
        lo, hi = mean - se, mean + se
        if sign == ">0":
            verdict = "supported" if lo > 0 else "contradicted" if hi < 0 else "indeterminate"
        else:
            verdict = "supported" if hi < 0 else "contradicted" if lo > 0 else "indeterminate"
        return {
            "mean": mean, "se": se,
            "n_pos": int((diffs > 0).sum()), "n_neg": int((diffs < 0).sum()),
            "verdict": verdict,
        }

    contrasts = {
        "P1": contrast("on-learned", "off", ">0"),
        "P2": contrast("on-oracle", "on-learned", "<0"),
        "P3": contrast("on-oracle", "off", "<0"),
    }

    for arm in ("off", "on-learned", "on-oracle"):
        print(f"{arm:>11s} : e_H moyen = {means[arm]:.4f}, controle positif {control[arm]}/5 learns")
    for name, c in contrasts.items():
        print(f"{name} : {c['mean']:+.4f} +/- {c['se']:.4f}  ({c['n_pos']}+/{c['n_neg']}-)  {c['verdict']}")


Les quinze cellules recalculées retombent, à l'arrondi près, sur `PROTOCOL.md` §28.7 :
$e_H$ moyen `off` **0.0702**, `on-learned` **0.2153**, `on-oracle` **0.0851** ; contrôle
positif `off` 5/5, `on-learned` 0/5, `on-oracle` 5/5 `learns` ; P1 **supported**
(+0.1451 ± 0.0351, 5+/0−), P2 **supported** (−0.1302 ± 0.0340, 0+/5−), P3
**indeterminate** (+0.0149 ± 0.0189, 3+/2−).

**P1 est le contrôle, pas un tirage indépendant.** Sa réplique à la décimale près du `mlp`
de §27.8.3 tient au fait que le pipeline est déterministe de bout en bout — mêmes graines,
même `data_hash` (`3cb3e669`), même ordonnancement, retiré entièrement depuis zéro dans
cette racine sous l'autosuffisance de §14.6. Elle valide la reproductibilité ; elle
n'ajoute aucune indépendance statistique, et se compte comme le contrôle que §28.3 a
déclaré, pas comme une découverte.


## 5. Le résultat

`PROTOCOL.md` §28.7 : **15/15 cellules complètes, zéro divergence**, un `gate.json` par
run.

| prédiction | moyenne ± SE | signes | verdict |
|---|---|---|---|
| P1 `on-learned − off > 0` | +0.1451 ± 0.0351 | 5+/0− | **supported** |
| P2 `on-oracle − on-learned < 0` | −0.1302 ± 0.0340 | 0+/5− | **supported** |
| P3 `on-oracle − off < 0` | +0.0149 ± 0.0189 | 3+/2− | **indeterminate** |

Le contrôle positif — publié à côté des nombres, **jamais utilisé pour l'éligibilité**
(§28.3, rappel d'une phrase du notebook 03 §4.3) : `off` 5/5, `on-learned` 0/5, `on-oracle`
5/5 `learns`.

C'est la seconde issue de §28.4 qui s'applique, et sa phrase est le rapport : *the coupling
is a contributing mechanism; it is not sufficient to reverse the sign*. Donner au bras
structuré la base vraie retire l'essentiel de la pénalité sur presque chaque graine (P2,
0+/5−) et rend au bras la porte de contrôle positif que le bras à base apprise échoue sur
les cinq graines (5/5 `learns` contre 0/5) ; cela ne met pas la classe structurée devant la
classe non structurée (P3 est à cheval sur zéro, 3+/2−). Cette phrase ne se reformule
**jamais** en « la structure marche quand on fixe la base » — §28.4 l'interdit
explicitement, et §27.8.1, §27.8.3 et §27.9 restent exactement ce qu'ils étaient, quelle
que soit l'issue de cette racine.

**Le coût, mesuré et non estimé** (§28.2) : une cellule `on-oracle` seule a tourné en
premier, isolée sur la machine — 2186 s (36 min 26 s), 0.273 s/mise à jour — avant que les
quatorze autres ne se lancent en trois lots parallèles.


## 6. La figure


In [ ]:
fig_out = REPO / "assets/figH_basis_oracle.png"

if results_root.exists():
    fig_path = make_figH(results_root, fig_out)
else:
    print(f"[garde] {results_root} absente -- affichage de la figure deja commitee dans assets/.")
    fig_path = fig_out

display(Image(str(fig_path)))


**Lecture.** Le panneau de gauche place les cinq graines par bras, moyenne ± 1 SE, avec
un trait fin reliant la même graine à travers les trois bras (§28.3 : chaque contraste est
apparié par graine, jamais par ordre de fichier ni par moyenne de groupe) ; le panneau de
droite trace les trois contrastes P1/P2/P3 eux-mêmes contre zéro. L'annotation rappelle que
`on-oracle` mesure un plafond, pas un concurrent loyal.

**Ce que la figure ne montre pas.** Elle ne montre aucun test de signification — §28.3
interdit toute valeur `p` — seulement des intervalles ±1 SE et des comptages de signe par
graine. Elle ne montre pas non plus que `on-oracle` serait atteignable sans connaître la
base vraie : c'est un plafond construit avec une information qu'aucun autre bras du dépôt
n'a le droit de voir.


## 7. Ce qui a été prouvé ici, la limite, et le pont

**Ce qui a été prouvé ici.** L'expérience que §26.0 a nommée et refusée sous le plafond de
§25.5 a été exécutée sous l'amendement de §28.0. La base gelée dans `basis_root.yaml`
rediagonalise le générateur vrai à 1.3e-15 et son sha256 a été vérifié ; un mutant qui
mélange les deux sous-espaces la casse à ≈0.1546, cinq ordres de grandeur au-dessus. Les
quinze verdicts ont été recalculés graine par graine et retombent sur `PROTOCOL.md` §28.7 :
P1 réplique le contrôle (déterminisme, pas un tirage indépendant), P2 est `supported`
(donner la base retire l'essentiel de la pénalité et restaure la porte de contrôle
positif), P3 est `indeterminate` (la classe structurée ne dépasse pas pour autant la classe
non structurée). La phrase liée de §28.4 tient : le couplage est un mécanisme contributif,
pas suffisant pour inverser le signe.

**Lien avec `sld`.** `src/sld/experiments/configs/basis_root.yaml` (le littéral $T$ et son
empreinte), `src/sld/systems/representability.py` (`representability_distance`,
`real_modal_basis`), `src/sld/viz/figH_basis_oracle.py` (`make_figH`),
`tests/test_basis_root.py`.

**La limite honnête.** `on-oracle` est un **plafond**, pas un concurrent loyal — sa base
lui est donnée, ce qu'aucun autre bras n'a le droit de voir. La portée tient sur un système
(`CoupledOscillators`), un backbone (`mlp`), une strate (§28.5) : elle ne mesure pas le
pendule, ne fait tourner ni le minimal causal transformer ni le SSM, ne fait varier ni `n`,
ni `h_fr`, ni `c`, ni le taux, ni le budget. Aucun seuil ne bouge.

**Le pont.** Sept racines pour poser le négatif, une pour en mesurer la cause, et une
figure de plus dans `assets/`. Le notebook 06 régénère chaque figure du dépôt depuis ses
propres artefacts — `figH_basis_oracle.png` y prend sa place aux côtés des huit autres.
